Summarize expression of known versus unknown BGCs.

In [171]:
import pandas as pd

In [172]:
file_path = '/Users/annasve/Desktop/article_data/antismash_table/all_antismash_regions_refined.csv'
mibig_df = pd.read_csv(file_path)

In [173]:
mibig_df.head()

,Strain,Region,Type,From,To,Most similar known cluster,Type MIBiG,Similarity
0,NBC_01734,1,terpene,"1,316,582","1,336,936",tetrachlorizine,Polyketide,13%
1,NBC_01734,2,"NRPS,betalactone","1,768,519","1,826,940",cyclomarin D,NRP,8%
2,NBC_01734,3,NAPAA,"1,929,187","1,963,104",ε-Poly-L-lysine,NRP,100%
3,NBC_01734,4,"RRE-containing,NRP-metallophore,NRPS,T1PKS","2,091,565","2,197,862",pyrrolomycin A/pyrrolomycin B/pyrrolomycin C/p...,Polyketide,34%
4,NBC_01734,5,lanthipeptide-class-iii,"2,255,111","2,277,708",SapB,RiPP:Lanthipeptide,100%


In [174]:
df = mibig_df.copy()

# Clean Region → keep only number
df["Region"] = (
    df["Region"]
    .astype(str)
    .str.extract(r"(\d+(?:_\d+)?)", expand=False)
)

# Normalize Type:
# - split on comma
# - strip spaces
# - sort alphabetically
# - join with "_"
df["Type_normalized"] = (
    df["Type"]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: "_".join(
            sorted(
                [part.strip() for part in x.split(",") if part.strip()],
                key=str.casefold
            )
        )
    )
)

# Create bgc_ID
df["BGC_ID"] = (
    df["Region"].astype(str)
    + "_"
    + df["Type_normalized"]
    + "_"
    + df["Strain"].astype(str)
)

# Save back
mibig_df = df

In [175]:
mibig_df.head()

,Strain,Region,Type,From,To,Most similar known cluster,Type MIBiG,Similarity,Type_normalized,BGC_ID
0,NBC_01734,1,terpene,"1,316,582","1,336,936",tetrachlorizine,Polyketide,13%,terpene,1_terpene_NBC_01734
1,NBC_01734,2,"NRPS,betalactone","1,768,519","1,826,940",cyclomarin D,NRP,8%,betalactone_NRPS,2_betalactone_NRPS_NBC_01734
2,NBC_01734,3,NAPAA,"1,929,187","1,963,104",ε-Poly-L-lysine,NRP,100%,NAPAA,3_NAPAA_NBC_01734
3,NBC_01734,4,"RRE-containing,NRP-metallophore,NRPS,T1PKS","2,091,565","2,197,862",pyrrolomycin A/pyrrolomycin B/pyrrolomycin C/p...,Polyketide,34%,NRP-metallophore_NRPS_RRE-containing_T1PKS,4_NRP-metallophore_NRPS_RRE-containing_T1PKS_N...
4,NBC_01734,5,lanthipeptide-class-iii,"2,255,111","2,277,708",SapB,RiPP:Lanthipeptide,100%,lanthipeptide-class-iii,5_lanthipeptide-class-iii_NBC_01734


In [176]:
mibig_df = mibig_df.fillna('None')

In [177]:
mibig_df.columns

Index(['Strain', 'Region', 'Type', 'From', 'To', 'Most similar known cluster',
       'Type MIBiG', 'Similarity', 'Type_normalized', 'BGC_ID'],
      dtype='object')

In [178]:
columns_to_keep = ['Most similar known cluster',
       'Type MIBiG', 'Similarity','BGC_ID']

In [179]:
mibig_merge = mibig_df[columns_to_keep]

In [180]:
save_path = '/Users/annasve/Desktop/article_data/antismash_table/all_antismash_regions_refined_BGC_ID.xlsx'
mibig_df.to_excel(save_path)

In [181]:
file_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_expression_antismash.xlsx'
bgc_expression = pd.read_excel(file_path)

In [182]:
bgc_expression

,Strain,BGC_ID,bgc_type,media_expressed
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,NRPS-like_nucleoside_other,not_expressed
1,NBC_00001,11_furan_NBC_00001,furan,"ISP2, gluc, gly, malt"
2,NBC_00001,12_NI-siderophore_NBC_00001,NI-siderophore,not_expressed
3,NBC_00001,13_nucleoside_NBC_00001,nucleoside,ISP2
4,NBC_00001,14_terpene_NBC_00001,terpene,SoyM
...,...,...,...,...
3988,NBC_01815,5_NI-siderophore_NBC_01815,NI-siderophore,"gluc, gly, malt"
3989,NBC_01815,6_T2PKS_NBC_01815,T2PKS,not_expressed
3990,NBC_01815,7_RiPP-like_NBC_01815,RiPP-like,"DNPM, SoyM, TSB"
3991,NBC_01815,8_terpene_NBC_01815,terpene,"DNPM, ISP2, MA, TSB, gluc"


In [183]:
bgc_expression = pd.merge(bgc_expression, mibig_merge, on = 'BGC_ID', how = 'left')

In [184]:
bgc_expression

,Strain,BGC_ID,bgc_type,media_expressed,Most similar known cluster,Type MIBiG,Similarity
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,NRPS-like_nucleoside_other,not_expressed,mitomycin,Other:Aminocoumarin,23%
1,NBC_00001,11_furan_NBC_00001,furan,"ISP2, gluc, gly, malt",None,None,None
2,NBC_00001,12_NI-siderophore_NBC_00001,NI-siderophore,not_expressed,None,None,None
3,NBC_00001,13_nucleoside_NBC_00001,nucleoside,ISP2,None,None,None
4,NBC_00001,14_terpene_NBC_00001,terpene,SoyM,albaflavenone,Terpene,100%
...,...,...,...,...,...,...,...
4015,NBC_01815,5_NI-siderophore_NBC_01815,NI-siderophore,"gluc, gly, malt",desferrioxamin B/desferrioxamine E,Other,83%
4016,NBC_01815,6_T2PKS_NBC_01815,T2PKS,not_expressed,spore pigment,Polyketide,83%
4017,NBC_01815,7_RiPP-like_NBC_01815,RiPP-like,"DNPM, SoyM, TSB",None,None,None
4018,NBC_01815,8_terpene_NBC_01815,terpene,"DNPM, ISP2, MA, TSB, gluc",albaflavenone,Terpene,100%


In [185]:
bgc_expression["BGC_ID"][bgc_expression["BGC_ID"].duplicated()]

154                       10_terpene_NBC_00077
156                          11_NRPS_NBC_00077
158                 12_butyrolactone_NBC_00077
160                      13_PKS-like_NBC_00077
162                 14_butyrolactone_NBC_00077
164        15_lanthipeptide-class-ii_NBC_00077
166                16_NI-siderophore_NBC_00077
168                17_NI-siderophore_NBC_00077
170                         18_T3PKS_NBC_00077
172                         19_other_NBC_00077
174                           1_NRPS_NBC_00077
176                       20_melanin_NBC_00077
178               21_ladderane_T2PKS_NBC_00077
180                22_RRE-containing_NBC_00077
182                23_NI-siderophore_NBC_00077
184            24_LAP_NRPS_NRPS-like_NBC_00077
186            25_NRPS_T1PKS_terpene_NBC_00077
188                       26_terpene_NBC_00077
190                       27_ectoine_NBC_00077
193                      2_NRPS-like_NBC_00077
195    3_lanthipeptide-class-ii_NRPS_NBC_00077
197          

In [186]:
bgc_expression = bgc_expression.drop_duplicates()

In [187]:
bgc_expression

,Strain,BGC_ID,bgc_type,media_expressed,Most similar known cluster,Type MIBiG,Similarity
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,NRPS-like_nucleoside_other,not_expressed,mitomycin,Other:Aminocoumarin,23%
1,NBC_00001,11_furan_NBC_00001,furan,"ISP2, gluc, gly, malt",None,None,None
2,NBC_00001,12_NI-siderophore_NBC_00001,NI-siderophore,not_expressed,None,None,None
3,NBC_00001,13_nucleoside_NBC_00001,nucleoside,ISP2,None,None,None
4,NBC_00001,14_terpene_NBC_00001,terpene,SoyM,albaflavenone,Terpene,100%
...,...,...,...,...,...,...,...
4015,NBC_01815,5_NI-siderophore_NBC_01815,NI-siderophore,"gluc, gly, malt",desferrioxamin B/desferrioxamine E,Other,83%
4016,NBC_01815,6_T2PKS_NBC_01815,T2PKS,not_expressed,spore pigment,Polyketide,83%
4017,NBC_01815,7_RiPP-like_NBC_01815,RiPP-like,"DNPM, SoyM, TSB",None,None,None
4018,NBC_01815,8_terpene_NBC_01815,terpene,"DNPM, ISP2, MA, TSB, gluc",albaflavenone,Terpene,100%


In [188]:
bgc_expression["BGC_ID"][bgc_expression["BGC_ID"].duplicated()]

Series([], Name: BGC_ID, dtype: object)

In [189]:
save_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_expression_antismash_mibig.xlsx'
bgc_expression.to_excel(save_path, index=False)